# Discord Bot LLM Fine-Tuning

Using code from: Shaw Talebi

github: https://github.com/ShawhinT/YouTube-Blog/blob/main/LLMs/model-compression/0_train-teacher.ipynb


### imports

In [ ]:
#!pip install datasets transformers evaluate

import datasets

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

### load data

In [ ]:
dataset_dict = load_dataset("csv", data_files="df_file_modified.csv")
dataset_dict["train"], dataset_dict["test"] = dataset_dict["train"].train_test_split(test_size=0.2).values()
dataset_dict["test"], dataset_dict["validation"] = dataset_dict["test"].train_test_split(test_size=0.5).values()

In [ ]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 729
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 91
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 92
    })
})

### Train Teacher Model

In [ ]:
# Load model directly
model_path = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_path)

id2label = {0: "sport", 1: "technology"}
label2id = {"sport": 0, "technology": 1}
model = AutoModelForSequenceClassification.from_pretrained(model_path,
                                                           num_labels=2,
                                                           id2label=id2label,
                                                           label2id=label2id,)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Freeze base model

In [ ]:
# print layers
for name, param in model.named_parameters():
   print(name, param.requires_grad)

bert.embeddings.word_embeddings.weight True
bert.embeddings.position_embeddings.weight True
bert.embeddings.token_type_embeddings.weight True
bert.embeddings.LayerNorm.weight True
bert.embeddings.LayerNorm.bias True
bert.encoder.layer.0.attention.self.query.weight True
bert.encoder.layer.0.attention.self.query.bias True
bert.encoder.layer.0.attention.self.key.weight True
bert.encoder.layer.0.attention.self.key.bias True
bert.encoder.layer.0.attention.self.value.weight True
bert.encoder.layer.0.attention.self.value.bias True
bert.encoder.layer.0.attention.output.dense.weight True
bert.encoder.layer.0.attention.output.dense.bias True
bert.encoder.layer.0.attention.output.LayerNorm.weight True
bert.encoder.layer.0.attention.output.LayerNorm.bias True
bert.encoder.layer.0.intermediate.dense.weight True
bert.encoder.layer.0.intermediate.dense.bias True
bert.encoder.layer.0.output.dense.weight True
bert.encoder.layer.0.output.dense.bias True
bert.encoder.layer.0.output.LayerNorm.weight True


In [ ]:
# freeze base model parameters
for name, param in model.base_model.named_parameters():
    param.requires_grad = False

# unfreeze base model pooling layers
for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad = True

In [ ]:
# print layers
for name, param in model.named_parameters():
   print(name, param.requires_grad)

bert.embeddings.word_embeddings.weight False
bert.embeddings.position_embeddings.weight False
bert.embeddings.token_type_embeddings.weight False
bert.embeddings.LayerNorm.weight False
bert.embeddings.LayerNorm.bias False
bert.encoder.layer.0.attention.self.query.weight False
bert.encoder.layer.0.attention.self.query.bias False
bert.encoder.layer.0.attention.self.key.weight False
bert.encoder.layer.0.attention.self.key.bias False
bert.encoder.layer.0.attention.self.value.weight False
bert.encoder.layer.0.attention.self.value.bias False
bert.encoder.layer.0.attention.output.dense.weight False
bert.encoder.layer.0.attention.output.dense.bias False
bert.encoder.layer.0.attention.output.LayerNorm.weight False
bert.encoder.layer.0.attention.output.LayerNorm.bias False
bert.encoder.layer.0.intermediate.dense.weight False
bert.encoder.layer.0.intermediate.dense.bias False
bert.encoder.layer.0.output.dense.weight False
bert.encoder.layer.0.output.dense.bias False
bert.encoder.layer.0.output.Lay

#### Preprocess text

In [ ]:
# define text preprocessing
def preprocess_function(examples):
    # Tokenize the input text and include the labels
    messages = [str(msg) for msg in examples["text"]] # Convert to string
    return tokenizer(messages, truncation=True, padding='max_length', max_length=128, return_tensors="pt")


# tokenize all dataset
tokenized_data = dataset_dict.map(preprocess_function, batched=True)

Map:   0%|          | 0/729 [00:00<?, ? examples/s]

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

Map:   0%|          | 0/92 [00:00<?, ? examples/s]

In [ ]:
# define a function to convert string labels to integers
def convert_labels(example):
    if example["labels"] == "sport":
        example["labels"] = 0
    elif example["labels"] == "technology":
        example["labels"] = 1
    return example

# Apply the function to the dataset before tokenization
dataset_dict = dataset_dict.map(convert_labels)

# tokenize all dataset
tokenized_data = dataset_dict.map(preprocess_function, batched=True)
tokenized_data = tokenized_data.cast_column("labels", datasets.features.ClassLabel(num_classes=2)) # We have 2 classes

Map:   0%|          | 0/729 [00:00<?, ? examples/s]

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

Map:   0%|          | 0/92 [00:00<?, ? examples/s]

Map:   0%|          | 0/729 [00:00<?, ? examples/s]

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

Map:   0%|          | 0/92 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/729 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

In [ ]:
# create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### Evaluation

In [ ]:
# load metrics
accuracy = evaluate.load("accuracy")
auc_score = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    # get predictions
    predictions, labels = eval_pred

    # apply softmax to get probabilities
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)
    # use probabilities of the positive class for ROC AUC
    positive_class_probs = probabilities[:, 1]
    # compute auc
    auc = np.round(auc_score.compute(prediction_scores=positive_class_probs, references=labels)['roc_auc'],3)

    # predict most probable class
    predicted_classes = np.argmax(predictions, axis=1)
    # compute accuracy
    acc = np.round(accuracy.compute(predictions=predicted_classes, references=labels)['accuracy'],3)

    return {"Accuracy": acc, "AUC": auc}

#### Train model

In [ ]:
# hyperparameters
lr = 2e-4
batch_size = 8
num_epochs = 10

training_args = TrainingArguments(
    output_dir="discord_classification2",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

<ipython-input-428-9d7aa9923850>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,0.148400,0.012506,1.000000,1.000000
2,0.016400,0.008582,1.000000,1.000000
3,0.013800,0.003768,1.000000,1.000000
4,0.007300,0.009456,0.989000,1.000000
5,0.005200,0.002292,1.000000,1.000000
6,0.006500,0.002059,1.000000,1.000000
7,0.004100,0.001845,1.000000,1.000000
8,0.006200,0.002292,1.000000,1.000000


Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,0.148400,0.012506,1.000000,1.000000
2,0.016400,0.008582,1.000000,1.000000
3,0.013800,0.003768,1.000000,1.000000
4,0.007300,0.009456,0.989000,1.000000
5,0.005200,0.002292,1.000000,1.000000
6,0.006500,0.002059,1.000000,1.000000
7,0.004100,0.001845,1.000000,1.000000
8,0.006200,0.002292,1.000000,1.000000
9,0.003200,0.002050,1.000000,1.000000
10,0.004400,0.002037,1.000000,1.000000


TrainOutput(global_step=920, training_loss=0.02156126408473305, metrics={'train_runtime': 3991.759, 'train_samples_per_second': 1.826, 'train_steps_per_second': 0.23, 'total_flos': 479519898393600.0, 'train_loss': 0.02156126408473305, 'epoch': 10.0})

### Apply Model to Validation Dataset

In [ ]:
# apply model to validation dataset
predictions = trainer.predict(tokenized_data["test"])

# Extract the logits and labels from the predictions object
logits = predictions.predictions
labels = predictions.label_ids

# Use your compute_metrics function
metrics = compute_metrics((logits, labels))
print(metrics)

{'Accuracy': 1.0, 'AUC': 1.0}


### Push to hub

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write

In [ ]:
# push model to hub
trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

events.out.tfevents.1734720667.26fea3e9f82a.205.25:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Ju1es/discord_classification2/commit/9f87b666ee10ea93d25f288b7cd049b353381284', commit_message='End of training', commit_description='', oid='9f87b666ee10ea93d25f288b7cd049b353381284', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ju1es/discord_classification2', endpoint='https://huggingface.co', repo_type='model', repo_id='Ju1es/discord_classification2'), pr_revision=None, pr_num=None)